> **Ekip paylaşım kopyası:** Kayıtlı çıktılar ve widget durumu dosya boyutunu azaltmak için temizlenmiştir; analiz hücreleri korunur. Veri dosyası Git deposuna dahil değildir. Kurulum, veri yolu ve mevcut veri dosyasını kopyalamadan kullanma seçenekleri için `../README.md` dosyasına bakın. `.venv` kernel'ini seçip Run All çalıştırın. Kaynak projedeki notebook ve çıktılar değiştirilmemiştir.

# DS02 — motor yaşam döngüsü ve bozulmanın keşifsel analizi
Yalnızca DS02 **dev** verisi kullanılır. Başlangıç motoru **2**. Önceki notebook'lar değiştirilmez. Motorun uçuşları artan `cycle` sırasıyla işlenir; bellekte tüm motor veya DS02 yerine bir uçuş ve küçük özet tabloları tutulur.

Bu çalışma model eğitimi, sensör seçimi, arıza başlangıcı kararı veya model preprocessing'i değildir. Ham sensör değişimleri çalışma koşullarından etkilenebilir. RUL'nin tanımı gereği azalması model başarısı değildir. `T_dev` gerçek sensör değil, simülasyonun iç sağlık durumudur; burada yalnızca keşif ve doğrulama amacıyla kullanılır. Parametreler doğrudan verim kaybı yüzdesi değildir.

**Kullanım:** VS Code'da proje `.venv` kernel'ini seçin ve Run All çalıştırın. En alttaki motor seçimi ve **Motoru analiz et** düğmesi diğer motorlar için aynı hesabı ve CSV kaydını yapar. Sensör seçimi özetlerden grafiği yeniler, HDF5'i tekrar okumaz.

**Sunum önceliği — Ana EDA:** Seçilen motorun bütün uçuşları ve bütün ilgili ölçümleri ana analiz kapsamındadır. RUL, sağlık parametreleri, ham sensör medyanları/Q1–Q3 ve çalışma koşullarının tam dağılım özetleri bu kapsamı temsil eder.

**İsteğe bağlı yardımcı inceleme:** Benzer koşul alt grubu, ana EDA'yı tamamlayan sınırlı bir karşılaştırmadır. Danışman sunumunda bu bölüm atlanabilir; mevcut hesaplama akışı ve kayıtlı grafikler korunmuştur. Alt gruba girmeyen ölçümler önemsiz, hatalı veya gereksiz değildir. Bunlar motorun farklı çalışma rejimlerini içerir ve ana EDA'da kullanılmaya devam eder.


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import ipywidgets as widgets
import os
import sys
# Matplotlib ?nbelle?i proje sanal ortam?nda tutulur.
os.environ.setdefault("MPLCONFIGDIR", str(Path(__import__("tempfile").gettempdir()) / "n_cmapss_matplotlib_cache"))
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Paylaşım kopyası: veri yolu bilgisayardan bağımsız çözülür.
import os

def resolve_ncmapss_paths(start=None):
    working_directory = Path.cwd().resolve() if start is None else Path(start).resolve()
    project_root = None
    for parent in [working_directory, *working_directory.parents]:
        if parent.name == "n_cmapss" and (parent / "notebooks").is_dir():
            project_root = parent
            break
        candidate = parent / "n_cmapss"
        if (candidate / "notebooks").is_dir():
            project_root = candidate
            break
    if project_root is None:
        raise FileNotFoundError("Notebook'u ortak repo, n_cmapss veya n_cmapss/notebooks klasöründen çalıştırın.")
    external_path = os.environ.get("N_CMAPSS_H5")
    h5_path = Path(external_path).expanduser().resolve() if external_path else project_root / "data" / "N-CMAPSS_DS02-006.h5"
    return project_root, h5_path

project_root, h5_path = resolve_ncmapss_paths()
if not h5_path.is_file():
    raise FileNotFoundError(
        f"Veri dosyası bulunamadı: {h5_path}. HDF5 repoya dahil değildir. "
        "Dosyayı yerel n_cmapss/data/ altında sağlayın veya kernel ortamında "
        "N_CMAPSS_H5 değişkenini mevcut dosyanın tam yoluna ayarlayın. README'ye bakın."
    )

DEV_KEYS = ["A_dev", "W_dev", "X_s_dev", "X_v_dev", "T_dev", "Y_dev"]
VAR_KEYS = ["A_var", "W_var", "X_s_var", "X_v_var", "T_var"]
CHUNK_ROWS = 100_000

def decode_names(dataset):
    return [v.decode("utf-8").rstrip("\x00") if isinstance(v, (bytes, np.bytes_)) else str(v)
            for v in np.asarray(dataset[()]).reshape(-1)]

with h5py.File(h5_path, "r") as h5:
    names = {key: decode_names(h5[key]) for key in VAR_KEYS}
    column_names = {key: names[var] for key, var in zip(DEV_KEYS[:-1], VAR_KEYS)}
    column_names["Y_dev"] = ["RUL"]
    row_counts = {key: h5[key].shape[0] for key in DEV_KEYS}
    if len(set(row_counts.values())) != 1:
        raise ValueError(f"Satır sayıları eşit değil: {row_counts}")
    for key in DEV_KEYS:
        if h5[key].ndim != 2 or h5[key].shape[1] != len(column_names[key]):
            raise ValueError(f"Sütun metadata uyumsuzluğu: {key}")
all_columns = [name for key in DEV_KEYS for name in column_names[key]]
assert len(all_columns) == len(set(all_columns)) == 47
unit_col = names["A_var"].index("unit")
cycle_col = names["A_var"].index("cycle")
print("dev satır sayısı:", row_counts["A_dev"])

%matplotlib inline
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.grid": True, "grid.alpha": 0.25})


## 1. Dev motor/uçuş kataloğu
A_dev unit/cycle sütunları 100.000 satırlık parçalarla taranır. Aynı satır aralıkları bütün dev dizilerinde kullanılır; sütun adları ilgili metadata dizilerinden alınır. Boş veya hizasız diziler sessizce birleştirilmez.

In [ ]:
flight_ranges = {}
with h5py.File(h5_path, "r") as h5:
    a = h5["A_dev"]
    for start in range(0, a.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, a.shape[0])
        units = a[start:stop, unit_col]
        cycles = a[start:stop, cycle_col]
        if not (np.isfinite(units).all() and np.isfinite(cycles).all()
                and np.equal(units, np.floor(units)).all()
                and np.equal(cycles, np.floor(cycles)).all()):
            raise ValueError("Motor/uçuş kimlikleri sonlu tam sayı olmalıdır")
        changes = np.flatnonzero((units[1:] != units[:-1]) | (cycles[1:] != cycles[:-1])) + 1
        boundaries = np.r_[0, changes, len(units)]
        for left, right in zip(boundaries[:-1], boundaries[1:]):
            key = (int(units[left]), int(cycles[left]))
            begin, end = start + int(left), start + int(right)
            ranges = flight_ranges.setdefault(key, [])
            if ranges and ranges[-1][1] == begin:
                ranges[-1] = (ranges[-1][0], end)
            else:
                ranges.append((begin, end))
engine_flights = {}
for unit, cycle in sorted(flight_ranges):
    engine_flights.setdefault(unit, []).append(cycle)
assert sum(end - begin for ranges in flight_ranges.values() for begin, end in ranges) == row_counts["A_dev"]
catalog = pd.DataFrame([
    {"unit": unit, "uçuş_sayısı": len(cycles), "ilk_cycle": min(cycles), "son_cycle": max(cycles)}
    for unit, cycles in engine_flights.items()
])
display(catalog)
print("Toplam motor/uçuş çifti:", len(flight_ranges))

assert list(engine_flights) == [2, 5, 10, 16, 18, 20]


## 2. Ana EDA: tüm ölçümlerden uçuş özetleri

Aşağıda hesaplanan ham özetler ana bilimsel sunumun temelidir.

### Ayrı yöntem notu: isteğe bağlı benzer koşul karşılaştırması
Her uçuşun **bütün ölçümleri** üzerinden sensör medyanı ve Q1–Q3; sağlık parametreleri için minimum, medyan, maksimum hesaplanır. Fc/hs değerlerinin tümü listelenir; uçuş içinde değişen değerler ayrıca raporlanır. RUL'nin sonlu ve uçuş içinde sabit olduğu kontrol edilir; tutarsızsa grafikte minimum–maksimum aralığı gösterilir.

**Karşılaştırma yöntemi:** seçilen motorun ilk uçuşundaki alt, Mach, TRA, T2 değişkenlerinin her biri için Q1–Q3 aralığı referans alınır. Bu dört aralığı **aynı anda** sağlayan gerçek ölçümler her uçuşta seçilir. Referans ilk uçuş olduğu için gelecekteki uçuşlardan eşik öğrenilmez. IQR merkezi dağılımı temsil eden açıklanabilir bir keşif tercihidir; tek doğru eşik veya koşul düzeltmesi değildir. Sabit koşulun IQR'si sıfırsa yalnızca eşit değerler eşleşir.

Ortak kutu boş kalabilir veya bazı uçuşlarda çok az örnek içerebilir. Örnek sayıları, oranları ve seçilmiş örneklerin koşul medyanları raporlanır. Boş uçuşlar NaN olarak bırakılır; tek örnekli uçuşlar ayrıca işaretlenir. İki örnek bile istatistiksel yeterlilik garantisi değildir. Aynı kutuda bulunmak koşul dağılımlarının eşit olduğu anlamına gelmez; dengeli eşleştirme, bağımsız örnekler veya nedensellik iddiası yoktur. Bu geçici alt grup kalıcı normalizasyon/ölçekleme değildir.

**Seçim yanlılığı:** İlk uçuşun merkezi koşul kutusunu referans almak, bu uçuşa benzeyen rejimleri daha fazla temsil eder. Tırmanış, alçalış, uç koşullar veya diğer çalışma rejimleri kutu dışında kalabilir; bu örnekler genel analizden çıkarılmaz. Uçuş başına eşleşme oranının farklı olması, yaşam döngüsü boyunca farklı rejim karışımlarının karşılaştırılmasına yol açabilir. Dolayısıyla alt grup sonucu bütün motor kullanımına genellenemez.

**Eşiklerin kaynağı:** `reference[conditions].quantile([0.25, 0.75])` ilk cycle'ın bütün ölçümlerinden, pandas'ın varsayılan doğrusal kuantil interpolasyonuyla hesaplanır. Dört koşulun alt/üst sınırları dahil edilir ve kesişim alınır. Merkezi %50 aralıklarını seçmek açıklanabilir fakat yine de bir yöntem tercihidir; ortak dört boyutlu kutunun %50 ölçüm içermesi beklenmez. Tam eşleştirme veya koşul etkilerini giderme sağlanmaz.


In [ ]:
sensors = names["X_s_var"] + names["X_v_var"]
physical_sensors = names["X_s_var"]
health_parameters = names["T_var"]
conditions = ["alt", "Mach", "TRA", "T2"]
assert len(sensors) == 28 and len(health_parameters) == 10
expected_health = ["fan_eff_mod", "fan_flow_mod", "LPC_eff_mod", "LPC_flow_mod", "HPC_eff_mod", "HPC_flow_mod", "HPT_eff_mod", "HPT_flow_mod", "LPT_eff_mod", "LPT_flow_mod"]
assert health_parameters == expected_health
assert set(conditions).issubset(names["W_var"])

def read_flight(h5, unit, cycle):
    frames = []
    for key in DEV_KEYS:
        blocks = [h5[key][begin:end, :] for begin, end in flight_ranges[(unit, cycle)]]
        array = blocks[0] if len(blocks) == 1 else np.concatenate(blocks, axis=0)
        frames.append(pd.DataFrame(array, columns=column_names[key]))
    data = pd.concat(frames, axis=1, verify_integrity=True)
    if not ((data.unit == unit).all() and (data.cycle == cycle).all()):
        raise ValueError(f"Satır hizalama/kimlik hatası: {unit}/{cycle}")
    if len(data) != sum(end-begin for begin, end in flight_ranges[(unit, cycle)]):
        raise ValueError("Ölçüm sayısı uyuşmazlığı")
    return data

def value_list(series):
    return ", ".join(str(v) for v in sorted(series.dropna().unique()))

def analyze_motor(unit):
    unit = int(unit)
    cycles = sorted(engine_flights[unit])
    rows, problems = [], []
    first_nonzero = {name: None for name in health_parameters}
    invalid_health = {name: 0 for name in health_parameters}
    with h5py.File(h5_path, "r") as h5:
        reference = read_flight(h5, unit, cycles[0])
        bounds = reference[conditions].quantile([0.25, 0.75])
        if not np.isfinite(bounds.to_numpy()).all():
            raise ValueError("Referans koşullar için sonlu Q1/Q3 hesaplanamadı")
        del reference
        for cycle in cycles:
            data = read_flight(h5, unit, cycle)
            row = {"unit": unit, "cycle": cycle, "n_samples": len(data)}
            finite = np.isfinite(data.to_numpy())
            row["nonfinite_count"] = int((~finite).sum())
            if row["nonfinite_count"]:
                problems.append(f"cycle {cycle}: {row['nonfinite_count']} sonlu olmayan değer; özetlerde NaN olarak dışlandı")
                data = data.where(np.isfinite(data), np.nan)
            rul = data.RUL
            row.update(RUL_first=rul.iloc[0], RUL_median=rul.median(), RUL_min=rul.min(), RUL_max=rul.max(),
                       RUL_constant=bool(rul.notna().all() and rul.nunique() == 1))
            if not row["RUL_constant"]:
                problems.append(f"cycle {cycle}: RUL uçuş içinde sabit/eksiksiz değil")
            for name in ["Fc", "hs"]:
                row[name + "_values"] = value_list(data[name])
                row[name + "_constant"] = bool(data[name].notna().all() and data[name].nunique() == 1)
                if not row[name + "_constant"]:
                    problems.append(f"cycle {cycle}: {name} uçuş içinde değişiyor veya eksik")
            mask = data[conditions].ge(bounds.loc[0.25]).all(axis=1) & data[conditions].le(bounds.loc[0.75]).all(axis=1)
            comparable = data.loc[mask, sensors + conditions]
            row["matched_n"] = len(comparable)
            row["matched_fraction"] = len(comparable) / len(data)
            for name in conditions + sensors:
                values = data[name]
                q = values.quantile([0.25, 0.5, 0.75])
                row.update({name + "_q1": q.loc[0.25], name + "_median": q.loc[0.5], name + "_q3": q.loc[0.75]})
                if name in conditions:
                    row[name + "_min"], row[name + "_max"] = values.min(), values.max()
                mq = comparable[name].quantile([0.25, 0.5, 0.75])
                row.update({name + "_matched_q1": mq.loc[0.25], name + "_matched_median": mq.loc[0.5], name + "_matched_q3": mq.loc[0.75]})
            for name in health_parameters:
                values = data[name]
                row.update({name + "_min": values.min(), name + "_median": values.median(), name + "_max": values.max()})
                nonzero = values.notna() & values.ne(0)
                row[name + "_nonzero_n"] = int(nonzero.sum())
                invalid_health[name] += int(values.isna().sum())
                if nonzero.any() and first_nonzero[name] is None:
                    first_nonzero[name] = cycle
            rows.append(row)
            del data, comparable
    summary = pd.DataFrame(rows).sort_values("cycle").reset_index(drop=True)
    health = pd.DataFrame([
        {"parameter": name, "nonzero_samples": int(summary[name + "_nonzero_n"].sum()),
         "first_nonzero_cycle": first_nonzero[name], "missing_samples": invalid_health[name],
         "status": "Sıfırdan farklı" if first_nonzero[name] is not None else
                   ("Sabit sıfır" if invalid_health[name] == 0 else "Eksik veri; sabit sıfır denemez")}
        for name in health_parameters])
    health["first_nonzero_cycle"] = health["first_nonzero_cycle"].astype("Int64")
    if (summary.RUL_median.diff().dropna() > 0).any():
        problems.append("RUL bazı ardışık uçuşlarda artıyor")
    if len(cycles) > 1 and np.any(np.diff(cycles) != 1):
        problems.append("Cycle dizisinde boşluklar var; olmayan uçuşlar doldurulmadı")
    return {"unit": unit, "summary": summary, "bounds": bounds, "health": health, "problems": problems}

def save_summaries(result):
    # Yeni çalıştırmalar önceki CSV'lerin üzerine yazmaz.
    from datetime import datetime
    from uuid import uuid4
    folder = project_root / "outputs" / f"DS02_dev_unit_{result['unit']}_lifecycle_{datetime.now():%Y%m%d_%H%M%S}_{uuid4().hex[:8]}"
    folder.mkdir(parents=True, exist_ok=False)
    result["summary"].to_csv(folder / "flight_summary.csv", index=False, encoding="utf-8-sig")
    result["health"].to_csv(folder / "health_parameters.csv", index=False, encoding="utf-8-sig")
    result["bounds"].rename_axis("quantile").to_csv(folder / "reference_condition_bounds.csv", encoding="utf-8-sig")
    pd.DataFrame({"bulgu": result["problems"] or ["RUL/Fc/hs/hizalama kontrollerinde beklenmeyen durum yok"]}).to_csv(folder / "checks.csv", index=False, encoding="utf-8-sig")
    result["csv_folder"] = folder
    return folder

def report(result):
    s = result["summary"]
    print(f"Motor {result['unit']}: {len(s)} uçuş; cycle {s.cycle.min()}–{s.cycle.max()}; {s.n_samples.sum():,} ölçüm")
    print(f"İlk/son uçuş RUL medyanı: {s.RUL_median.iloc[0]} / {s.RUL_median.iloc[-1]}")
    print(f"RUL sabit ve eksiksiz uçuşlar: {int(s.RUL_constant.sum())}/{len(s)}")
    print(f"Karşılaştırılabilir alt grup: {int(s.matched_n.sum()):,} ölçüm; {int((s.matched_n>0).sum())}/{len(s)} uçuşta eşleşme")
    print("Eşleşen örnek sayısı aralığı:", int(s.matched_n.min()), "–", int(s.matched_n.max()))
    print("Eşleşmeyen cycle'lar:", s.loc[s.matched_n == 0, "cycle"].tolist())
    print("Tek örnekli cycle'lar:", s.loc[s.matched_n == 1, "cycle"].tolist())
    for problem in result["problems"]:
        print("Kontrol bulgusu:", problem)
    print("CSV klasörü:", result.get("csv_folder", "Henüz kaydedilmedi"))
    display(result["health"])
    display(result["bounds"].rename(index={0.25: "Referans Q1", 0.75: "Referans Q3"}))
    # Her uçuş bir satır; çalışma koşullarının daha ayrıntılı istatistikleri CSV'de.
    columns = ["unit", "cycle", "n_samples", "RUL_first", "RUL_min", "RUL_max", "RUL_constant", "Fc_values", "hs_values", "matched_n", "matched_fraction"] + [name+"_median" for name in conditions]
    table_html = s[columns].to_html(index=False, max_rows=None)
    display(widgets.HTML(value="<div style='height:420px;overflow:scroll;width:100%'>"+table_html+"</div>", layout=widgets.Layout(width="1000px", max_width="100%")))


## 3. Yaşam döngüsü grafikleri
RUL, iç sağlık parametreleri, sensörler ve koşullar ayrı grafiklerde gösterilir. Her sensör kendi Y eksenine sahiptir; fiziksel sensörler mavi, sanal sensörler turuncudur. Çizgi uçuş medyanını, bant uçuş içi Q1–Q3 dağılımını gösterir; bant güven aralığı değildir.

Sağlık parametrelerinin tümü için panel vardır; sürekli sıfır panellerine çizgi eklemek yerine durum yazılır. Aktif paneller medyan ve min–maksimum aralığını içerir. **İlk tam sıfırdan farklı ölçüm**, belirgin bozulma başlangıcı değildir; toleranslı bir eşik veya arıza kararı uygulanmaz. Birimler metadata içinde doğrulanmadığı için sensörlere tahmini birim atanmamıştır; Y eksenleri özgün veri değeridir.

**Grafiklerin okuma sırası:** Önce RUL, T_dev, 28 ham sensör paneli ve koşul grafiğinin “bütün uçuş örnekleri” sütunu incelenir: bunlar **ana EDA**'dır. “Ortak koşul kutusu”, “benzer koşul” ve eşleşen örnek sayısı panelleri ise **isteğe bağlı yardımcı karşılaştırma**'dır. Hizalı sensör/sağlık grafiklerinde de ham ve alt grup panelleri bu ayrı kapsamlarla okunmalıdır; alt grup paneli ham panelin yerine geçmez.


In [ ]:
X_LABEL = "cycle — uçuş döngüsü"
def finish(fig):
    try:
        display(fig)
    finally:
        plt.close(fig)

def band_plot(ax, s, name, matched=False, color="tab:blue"):
    prefix = name + ("_matched" if matched else "")
    ax.plot(s.cycle, s[prefix+"_median"], color=color, linewidth=1)
    ax.fill_between(s.cycle.to_numpy(), s[prefix+"_q1"].to_numpy(), s[prefix+"_q3"].to_numpy(), color=color, alpha=0.16)
    ax.set_ylabel(f"{name} — özgün değer", fontsize=9)
    ax.tick_params(labelsize=8)

def rul_figure(result):
    s=result["summary"]
    fig,ax=plt.subplots(figsize=(12,4), layout="constrained")
    ax.plot(s.cycle,s.RUL_median,marker=".",label="Uçuş RUL medyanı")
    if not s.RUL_constant.all():
        ax.fill_between(s.cycle,s.RUL_min,s.RUL_max,alpha=.2,label="Uçuş içi min–maks.")
    ax.set(title=f"Motor {result['unit']} — RUL (tanım gereği azalma)",xlabel=X_LABEL,ylabel="RUL — kalan uçuş döngüsü")
    ax.legend(); return fig

def health_figure(result):
    s=result["summary"]; health=result["health"].set_index("parameter")
    fig,axes=plt.subplots(5,2,figsize=(15,15),sharex=True,sharey=False,layout="constrained")
    for ax,name in zip(axes.flat,health_parameters):
        ax.set_title(name)
        if health.loc[name,"nonzero_samples"] == 0:
            ax.text(.5,.5,health.loc[name,"status"],ha="center",va="center",transform=ax.transAxes)
        else:
            ax.plot(s.cycle,s[name+"_median"],color="tab:red")
            ax.fill_between(s.cycle,s[name+"_min"],s[name+"_max"],color="tab:red",alpha=.2)
        ax.set_ylabel("İç sağlık parametresi değeri",fontsize=9)
    fig.suptitle(f"Motor {result['unit']} — T_dev: medyan ve min–maks. (sensör değil)")
    fig.supxlabel(X_LABEL); return fig

def sensors_figure(result):
    s=result["summary"]
    fig,axes=plt.subplots(7,4,figsize=(19,22),sharex=True,sharey=False,layout="constrained")
    for ax,name in zip(axes.flat,sensors):
        physical=name in physical_sensors
        band_plot(ax,s,name,color="tab:blue" if physical else "tab:orange")
        ax.set_title(f"{name} ({'fiziksel' if physical else 'sanal'})",fontsize=11)
    fig.suptitle(f"Motor {result['unit']} — ham sensör medyanları ve uçuş içi Q1–Q3")
    fig.supxlabel(X_LABEL); return fig

def conditions_figure(result):
    s=result["summary"]
    fig,axes=plt.subplots(4,2,figsize=(16,12),sharex=True,sharey=False,layout="constrained")
    for row,name in enumerate(conditions):
        band_plot(axes[row,0],s,name,color="tab:green")
        band_plot(axes[row,1],s,name,matched=True,color="tab:purple")
        axes[row,0].set_title(f"{name} — bütün uçuş örnekleri")
        axes[row,1].set_title(f"{name} — ortak koşul kutusundaki örnekler")
    fig.suptitle(f"Motor {result['unit']} — koşul medyanları ve Q1–Q3; sağdaki dağılımlar da eşit olmayabilir")
    fig.supxlabel(X_LABEL); return fig

def comparison_figure(result,name):
    s=result["summary"]
    fig,axes=plt.subplots(3,1,figsize=(13,10),sharex=True,sharey=False,layout="constrained")
    color="tab:blue" if name in physical_sensors else "tab:orange"
    band_plot(axes[0],s,name,color=color)
    band_plot(axes[1],s,name,matched=True,color=color)
    axes[0].set_title(f"{name} — ham uçuş medyanı ve Q1–Q3")
    axes[1].set_title(f"{name} — benzer koşul kutusunda medyan ve Q1–Q3 (eksik uçuşlar boş)")
    axes[2].bar(s.cycle,s.matched_n,color="tab:gray")
    axes[2].set(ylabel="Eşleşen örnek sayısı",xlabel=X_LABEL)
    fig.suptitle(f"Motor {result['unit']} — koşul karşılaştırması; bantlar güven aralığı değildir")
    return fig

def relationship_figure(result,name):
    s=result["summary"]
    active=result["health"].loc[result["health"].nonzero_samples>0,"parameter"].tolist()
    fig,axes=plt.subplots(2+len(active),1,figsize=(13,3*(2+len(active))),sharex=True,sharey=False,layout="constrained",squeeze=False)
    axes=axes[:,0]
    band_plot(axes[0],s,name,color="tab:blue"); axes[0].set_title(f"{name} — ham medyan / Q1–Q3")
    band_plot(axes[1],s,name,matched=True,color="tab:orange"); axes[1].set_title(f"{name} — benzer koşullarda medyan / Q1–Q3")
    for ax,param in zip(axes[2:],active):
        ax.plot(s.cycle,s[param+"_median"],color="tab:red")
        ax.fill_between(s.cycle,s[param+"_min"],s[param+"_max"],alpha=.2,color="tab:red")
        ax.set_title(param+" — simülasyon iç sağlık durumu")
        ax.set_ylabel("Parametre değeri")
    axes[-1].set_xlabel(X_LABEL)
    fig.suptitle(f"Motor {result['unit']} — hizalı bağımsız Y eksenleri; ortak eğilim nedensellik göstermez")
    return fig


## 4. Motor 2: hesaplama, kontroller, CSV ve başlangıç grafikleri
Fc/hs değişimleri raporlanır; bunlar otomatik veri hatası sayılmaz. Hs bir uçuş içinde değişiyorsa listede bütün gözlenen değerler bulunur. Sonraki uçuşlardaki koşulların etkisi grafiklerle kesin olarak ayrılamaz. Burada aktif T parametrelerinin hepsi seçilen sensörün altına çizilir; belirli bir sensör–parametre nedensel eşleştirmesi varsayılmaz.

In [ ]:
result = analyze_motor(2)
s = result["summary"]
assert len(s) == len(engine_flights[2]) == 75
assert s.n_samples.sum() == sum(end-begin for (unit,cycle),ranges in flight_ranges.items() if unit==2 for begin,end in ranges)
assert s.cycle.is_monotonic_increasing and s.cycle.is_unique
assert s.RUL_constant.all()
assert s.RUL_first.iloc[0] == 74 and s.RUL_first.iloc[-1] == 0
assert np.array_equal(s.RUL_median.to_numpy(), 75-s.cycle.to_numpy())
assert (s.matched_n <= s.n_samples).all()
assert (s.nonfinite_count == 0).all()
save_summaries(result)
report(result)
finish(rul_figure(result))
finish(health_figure(result))
finish(sensors_figure(result))
finish(conditions_figure(result))
finish(comparison_figure(result,sensors[0]))
finish(relationship_figure(result,sensors[0]))


## Motor 2 için çalıştırılmış veriden doğrulanan sonuçlar
- 75 uçuş (cycle 1–75), toplam **853.142 ölçüm**. Altı dev dizisinin bütün uçuşlardaki satır hizalaması kaynakla karşılaştırıldı.
- RUL bütün 75 uçuşta uçuş içinde sabittir; ilk uçuşta 74, son uçuşta 0. Bu azalma hedefin tanımıyla uyumludur; bir sensör keşfi veya model başarısı değildir.
- **HPT_eff_mod** bütün ölçümlerde sıfırdan farklıdır; ilk sıfırdan farklı cycle **1**. Diğer dokuz iç sağlık parametresi sabit sıfırdır. İlk uçuşta sıfırdan farklı olmak belirgin bozulma başlangıcını belirlemez.
- Fc bütün uçuşlarda 3; hs uçuşlar arasında 1 ve 0 değerlerini alır. Uçuş içi Fc/hs ve RUL kontrollerinde beklenmeyen durum bulunmadı; sonlu olmayan değer yok.
- İlk uçuşun koşul IQR kutusunda **268.618 ölçüm** ve **71/75 uçuş** temsil edilir. Cycle **13, 24, 31, 63** için eşleşme yoktur; bu uçuşların benzer koşul sensör istatistikleri doldurulmadı. Eşleşen örnek sayısı uçuş başına 0–10.468 arasında değişir.
- Ham ve alt grup koşul dağılımları uçuşlar arasında değişmektedir. Alt grup uçuşları tam olarak dengelemez; sensör değişimini yalnızca bozulmaya bağlamak mümkün değildir. Kesin sensör seçimi, nedensellik veya arıza başlangıcı sonucu çıkarılmadı.

Bu bulgular yalnızca ilk çalıştırılan **Motor 2** içindir. Başka motor seçildiğinde alt bölümde yeni motorun raporu ve CSV dosyaları oluşur; bu kayıtlı açıklama Motor 2 referansı olarak kalır. İstatistikler ilgili uçuşun bütün ölçümlerinden hesaplandı; özet hesabında örnekleme uygulanmadı.


**Kapsamın korunması:** Cycle 13, 24, 31 ve 63 yalnızca bu referans kutusundaki yardımcı karşılaştırmada temsil edilememiştir. Bu dört uçuşun bütün ölçümleri ana EDA'da, ham sensör özetlerinde, RUL/sağlık/koşul grafiklerinde ve CSV'de yer alır. Genel analizden uçuş veya ölçüm çıkarılmamıştır.

**Ham özet ve ayrıntı doğrulaması:** Bu notebook'taki ham sensör medyanı ve Q1–Q3, `data[name].quantile([0.25, 0.5, 0.75])` ile ilgili uçuşun bütün satırlarından hesaplanır; yalnızca `_matched_` isimli özetler alt gruba aittir. Motor 2'de sonlu olmayan değer bulunmadığı için ham özetlerden bu nedenle dışlanan ölçüm de yoktur. Uçuş içi ayrıntılı sensör grafikleri **03_DS02_Sensor_Analysis.ipynb** içindedir; `label_axis` fonksiyonu seçilen uçuşun bütün `sample_index` ve sensör değerlerini çizer. 04'ün ayrıntılı grafikleri ise bütün ölçümlerden hesaplanan **uçuş başına özetleri** çizer; tek tek uçuş içi örnek grafikleri değildir.


## Ana EDA kapsam denetimi: her uçuşun toplam ve seçilen ölçüm sayısı
Bu tablo mevcut `result["summary"]` özetlerini görüntüler; HDF5 yeniden okunmaz ve hiçbir istatistik değiştirilmez. **Genel analizde** sütunu bütün uçuşlar için doğrudur. **Alt grup dışındaki** sayısı veri silindiği anlamına gelmez. Motor 2 için 75 uçuşun tamamı tabloda gösterilir. Sonuç başka motor için hesaplandıktan sonra bu kontrol hücrelerini tekrar çalıştırabilirsiniz.

In [ ]:
# Yalnızca mevcut özetleri görüntüleyen kontrol tablosu; veri eleme yok.
from IPython.display import HTML
eda_summary = result["summary"]
coverage_table = eda_summary[["unit", "cycle", "n_samples", "matched_n"]].copy()
coverage_table["alt_grup_dışındaki"] = coverage_table.n_samples - coverage_table.matched_n
coverage_table["seçilen_yüzde"] = 100 * coverage_table.matched_n / coverage_table.n_samples
coverage_table["genel_analizde"] = True
coverage_table["yardımcı_karşılaştırmada"] = coverage_table.matched_n > 0
coverage_table = coverage_table.rename(columns={"n_samples": "toplam_ölçüm", "matched_n": "alt_gruba_seçilen"})
assert len(coverage_table) == len(eda_summary)
assert (coverage_table.alt_gruba_seçilen <= coverage_table.toplam_ölçüm).all()
print(f"Motor {result['unit']}: {len(coverage_table)} uçuşun tamamı ana EDA'da.")
print(f"Toplam: {coverage_table.toplam_ölçüm.sum():,}; alt grup: {coverage_table.alt_gruba_seçilen.sum():,}; alt grup dışında: {coverage_table.alt_grup_dışındaki.sum():,}")
print("Yalnızca yardımcı karşılaştırmada temsil edilmeyen cycle'lar:", coverage_table.loc[~coverage_table.yardımcı_karşılaştırmada, "cycle"].tolist())
display(HTML("<div style='width:100%;height:450px;overflow:scroll'>" + coverage_table.to_html(index=False, max_rows=None, float_format=lambda value: f"{value:.2f}") + "</div>"))


## İsteğe bağlı yardımcı kontrol: seçilen koşullar gerçekten ne kadar benzer?
Aşağıdaki tablolar aynı kutuya girmiş örneklerin **eşit koşullara sahip olduğunu varsaymaz**. İlk tablo referans Q1/Q3 sınırlarını ve uçuşlar arasında seçilmiş koşul medyanlarının aralığını verir. “Medyan aralığı / referans IQR”, koşulların uçuşlar arası kaymasının referans aralığına oranıdır; 0 tam medyan eşitliğini, 1 kutu genişliği kadar farkı ifade eder. Bu bir benzerlik skoru, nedensellik testi veya kabul eşiği değildir. Referans IQR sıfırsa oran tanımsız bırakılır.

İkinci tablo her uçuşun seçilmiş örnek sayısını ve dört koşul için seçilmiş Q1/medyan/Q3 değerlerini listeler. Örtüşme ve dağılım farkları bu değerlerle ve mevcut koşul grafikleriyle incelenebilir; özet kuantiller çok değişkenli dağılım eşitliğini doğrulamaz. Eşleşme bulunmayan uçuşlar NaN kalır. Ham özetler, grafikler ve bütün uçuşlar korunur.

In [ ]:
# Mevcut koşul özetlerinden tanımlayıcı benzerlik raporu; yeni eşik uygulanmaz.
condition_rows = []
for name in conditions:
    lower, upper = result["bounds"].loc[0.25, name], result["bounds"].loc[0.75, name]
    medians = eda_summary[name + "_matched_median"].dropna()
    span = medians.max() - medians.min() if len(medians) else np.nan
    condition_rows.append({
        "koşul": name, "referans_Q1": lower, "referans_Q3": upper,
        "referans_IQR": upper-lower, "temsil_edilen_uçuş": len(medians),
        "seçilen_medyan_min": medians.min(), "seçilen_medyan_max": medians.max(),
        "medyan_aralığı_referans_IQR_oranı": span/(upper-lower) if upper>lower else np.nan,
    })
condition_similarity_table = pd.DataFrame(condition_rows)
display(condition_similarity_table)
selected_condition_table = eda_summary[["unit", "cycle", "matched_n", "matched_fraction"] + [name + "_matched_" + stat for name in conditions for stat in ["q1", "median", "q3"]]].copy()
display(HTML("<div style='width:100%;height:450px;overflow:scroll'>" + selected_condition_table.to_html(index=False, max_rows=None, float_format=lambda value: f"{value:.6g}") + "</div>"))
print("Yalnızca tanımlayıcı kontrol: kutu içi koşul farkları, seçim yanlılığı ve uçuş içi örnek bağımlılığı devam eder.")


## 5. Etkileşimli motor ve sensör seçimi
Yeni motor seçimi önceki etkileşimli görünümü temizler. **Motoru analiz et** bütün uçuş özetlerini hesaplar ve yeni CSV klasörüne kaydeder; sonuçlar alt alanda gösterilir. Sensör seçimi yalnızca son hesaplanan özetlerden iki ayrıntılı grafiği yeniler. Motor 2'nin başlangıç grafikleri yukarıda kayıtlıdır.

**Yorum sınırları:** bütün motor medyanlarında uçuş koşullarının etkisi vardır. Benzer koşul kutusu yalnızca kısmi karşılaştırılabilirlik sağlar; diğer koşullar, kutu içi dağılım farkları ve uçuş içi bağımlılık devam eder. İlk sıfırdan farklı sağlık değeri, istatistiksel veya fiziksel arıza başlangıcı değildir. Kesin sensör/özellik seçimi yapılmaz.

In [ ]:
engine = widgets.Dropdown(options=list(engine_flights),value=2,description="Motor:")
sensor = widgets.Dropdown(options=[(f"{name} — {'fiziksel' if name in physical_sensors else 'sanal'}",name) for name in sensors],description="Sensör:")
button = widgets.Button(description="Motoru analiz et",button_style="primary")
status = widgets.HTML(value="Motor 2 sonuçları yukarıda kayıtlıdır.")
general_output, detail_output = widgets.Output(), widgets.Output()
state = {"result": result}

def clear_views():
    for output in [general_output,detail_output]:
        with output: clear_output(wait=False)

def engine_changed(change):
    state["result"] = None
    clear_views()
    status.value = "Seçilen motor için Motoru analiz et düğmesine basın."

def redraw_detail(change=None):
    current=state["result"]
    if current is None: return
    with detail_output:
        clear_output(wait=True)
        finish(comparison_figure(current,sensor.value))
        finish(relationship_figure(current,sensor.value))

def on_analyze(clicked):
    button.disabled=True
    state["result"]=None
    clear_views()
    status.value="Uçuşlar sırayla işleniyor..."
    try:
        current=analyze_motor(engine.value)
        save_summaries(current)
        state["result"]=current
        with general_output:
            report(current)
            finish(rul_figure(current))
            finish(health_figure(current))
            finish(sensors_figure(current))
            finish(conditions_figure(current))
        redraw_detail()
        status.value=f"Motor {current['unit']} hazır; özet CSV dosyaları outputs altında."
    except Exception as exc:
        status.value="Analiz tamamlanamadı."
        with general_output: print(f"Hata: {exc}")
    finally:
        button.disabled=False

engine.observe(engine_changed,names="value")
sensor.observe(redraw_detail,names="value")
button.on_click(on_analyze)
display(widgets.VBox([widgets.HBox([engine,button]),sensor,status,general_output,detail_output]))
